# Ghost-Frequency GPU Validation

Before spending IBM QPU time on the ghost-frequency Ramsey campaign, we validate
the experimental design by classical simulation:

1. **Null check** — does the standard noise model (T1, Hahn-echo T2, readout)
   predict ~zero phase drift? (It should.)
2. **Ghost check** — does a quasi-static detuning δ ≈ 30 kHz reproduce the
   observed ~1 rad drift over ~4.4 µs? (It should.)
3. **Power analysis** — how many shots to resolve 20 kHz from 0 at 5σ?
4. **Mimic check** — can pulse-amplitude miscalibration fake the signature?
5. **Crosstalk check** — does a ZZ model give a spectator-dependent shift
   (distinguishable from intrinsic detuning)?

Observed anchors (novel-insights hunt, 2026-10-15):
- Kingston: −1.130 rad over 4.392 µs → −40.9 kHz
- Fez N21: +0.59 rad over 4.75 µs → +19.8 kHz
- Branch-independent on Kingston (rules out branch-dependent ZZ there).


In [ ]:
import torch, numpy as np, math
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
rng = np.random.default_rng(0)
torch.manual_seed(0)


## Observed anchors and device parameters
Typical Heron calibration values (medians; per-qubit values not retained):

In [ ]:
# Observed phase drifts
OBS = {
    'kingston': {'phase': -1.130, 't_us': 4.392},   # -> -40.9 kHz
    'fez_n21':  {'phase': +0.59,  't_us': 4.75},    # -> +19.8 kHz
}
for k, v in OBS.items():
    det_khz = v['phase'] / (2*math.pi*v['t_us']*1e-6) / 1e3
    print(f"{k}: implied detuning = {det_khz:+.1f} kHz")

# Representative Heron noise parameters
T1_us = 200.0        # energy relaxation
T2_echo_us = 150.0   # Hahn-echo T2 (refocuses quasi-static detuning!)
T2_star_us = 80.0    # Ramsey T2* (includes quasi-static contributions)
readout_err = 0.02   # per-qubit readout infidelity
DELAYS_us = np.array([1.0, 2.0, 4.0, 8.0])  # spec delays


## 1–2. Analytical Ramsey model: null vs ghost

Ramsey: X/2 – idle(t) – X/2 – measure.
With detuning δ and dephasing T2*: P(1) = (1 + e^(−t/T2*) · cos(2πδt)) / 2.

Null (H0): δ = 0. Ghost (H1): δ = 30 kHz (midpoint of 20–40 kHz band).

In [ ]:
def ramsey_p1(t_us, delta_khz, t2star_us=T2_star_us, vis=1.0):
    t = np.asarray(t_us, dtype=float)
    phase = 2*np.pi*delta_khz*1e3*t*1e-6
    return 0.5*(1.0 + vis*np.exp(-t/t2star_us)*np.cos(phase))

print(f"{'t (us)':>7} {'null P(1)':>10} {'ghost P(1)':>11} {'ghost phase (rad)':>17}")
for t in [4.392, 4.75]:
    p_null = ramsey_p1(t, 0.0)
    p_ghost = ramsey_p1(t, 30.0)
    ph = 2*np.pi*30e3*t*1e-6
    print(f"{t:>7.3f} {p_null:>10.4f} {p_ghost:>11.4f} {ph:>17.3f}")

# Null predicts NO systematic phase drift; ghost predicts ~0.83-0.90 rad
# at the observed idle times — matching the observed ~0.6-1.1 rad.


## 3. Power analysis (GPU Monte Carlo)

Question: with N shots per delay, can we resolve δ = 20 kHz from δ = 0 at 5σ?
We simulate Bernoulli shot noise on a GPU tensor grid: (N_shots × delays × trials).
Phase estimator: φ̂ = arccos(clip(2·p̂ − 1, −1, 1)) at the longest delay.

In [ ]:
def phase_uncertainty(n_shots, t_us=8.0, delta_khz=20.0, trials=2000):
    p = ramsey_p1(t_us, delta_khz)
    # GPU: binomial sampling via Bernoulli trials
    shots = torch.rand(trials, n_shots, device=device) < p
    p_hat = shots.float().mean(dim=1).cpu().numpy()
    vis = np.exp(-t_us/T2_star_us)
    cos_est = np.clip((2*p_hat - 1)/vis, -1, 1)
    return np.std(np.arccos(cos_est))

target_phase = 2*np.pi*20e3*8e-6   # 20 kHz at 8 us
print(f'target phase for 20 kHz @ 8us: {target_phase:.3f} rad; 5-sigma needs sigma < {target_phase/5:.3f}')
for n in [200, 500, 1000, 2000, 4000]:
    s = phase_uncertainty(n)
    print(f'N={n:5d}: sigma_phase={s:.4f} rad  ->  {target_phase/s:.1f} sigma')


## 4. Mimic check: pulse-amplitude miscalibration

If the X/2 pulses are miscalibrated by angle ε, fringe *visibility* drops
but the phase-vs-delay stays flat — a distinguishable signature from a
true detuning (which *drifts* phase linearly with t).

In [ ]:
def ramsey_p1_miscal(t_us, eps_rad, delta_khz=0.0):
    # X/2 -> rotation by pi/2 + eps; visibility scales as cos^2(eps)-ish
    vis = np.cos(eps_rad)**2
    return ramsey_p1(t_us, delta_khz, vis=vis)

t = np.linspace(0, 8, 9)
print('Miscalibrated (eps=0.2 rad), NO detuning:')
for ti, p in zip(t, ramsey_p1_miscal(t, 0.2)):
    print(f'  t={ti:.1f}us  P(1)={p:.4f}')
print('True detuning 30 kHz, perfect pulses:')
for ti, p in zip(t, ramsey_p1(t, 30.0)):
    print(f'  t={ti:.1f}us  P(1)={p:.4f}')
print()
print('Verdict: miscalibration suppresses contrast uniformly; detuning OSCILLATES.')
print('Fitting phase-vs-t separates them cleanly — miscalibration cannot mimic H1.')


## 5. Crosstalk check: static ZZ vs intrinsic detuning

A ZZ coupling ζ between work and spectator qubit adds phase 2πζt *only when
the spectator is in |1⟩. Intrinsic detuning is spectator-independent.
Experiment A (|0⟩ vs |1⟩ spectators) separates them.

In [ ]:
def ramsey_p1_zz(t_us, zz_khz, spectator_one):
    d = zz_khz if spectator_one else 0.0
    return ramsey_p1(t_us, d)

zz = 25.0  # kHz of ZZ coupling
print(f'ZZ = {zz} kHz:')
for t in DELAYS_us:
    p0 = ramsey_p1_zz(t, zz, False); p1 = ramsey_p1_zz(t, zz, True)
    print(f'  t={t:.0f}us  spectator|0>: P(1)={p0:.4f}   spectator|1>: P(1)={p1:.4f}')
print()
print('Verdict: ZZ gives a spectator-DEPENDENT shift.')
print('If H1 (intrinsic 20-40 kHz) is real, |0> vs |1> spectators give the SAME detuning.')
print('Experiment A falsifies ZZ-confusion directly.')


## Verdict

Fill in from the outputs above:
- Null predicts flat phase; ghost (30 kHz) reproduces the observed drift. **Design sane.**
- Shot budget for 5σ resolution of 20 kHz: read from §3.
- Amplitude miscalibration **cannot** mimic the drifting-phase signature.
- ZZ crosstalk **is** distinguishable via the spectator |0⟩/|1⟩ comparison.

If all checks pass, the QPU campaign (spec §3) is worth the ~120 s budget:
isolated controls first (cheapest falsification), full-chip checkerboard second.